In [1]:
# If needed:
# pip install qiskit qiskit-algorithms qiskit-optimization qiskit-aer

import numpy as np

from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer


# -----------------------------
# 1. Build a 10-variable binary optimization problem
# -----------------------------
qp = QuadraticProgram("toy_insurance_qaoa_10")

n = 10
for i in range(n):
    qp.binary_var(name=f"x{i}")

# Toy "coverage values" (you can replace these with insurance scores, margins, etc.)
values = [8, 6, 5, 9, 7, 4, 10, 3, 6, 8]

# Maximize sum(values_i * x_i)
qp.maximize(linear={f"x{i}": values[i] for i in range(n)})

# Constraint: choose exactly 3 coverages
qp.linear_constraint(
    linear={f"x{i}": 1 for i in range(n)},
    sense="==",
    rhs=3,
    name="pick_3"
)

print("Quadratic Program:")
print(qp.prettyprint())


# -----------------------------
# 2. Set up QAOA
# -----------------------------
sampler = StatevectorSampler()
optimizer = COBYLA(maxiter=200)

# reps = p in QAOA language; larger reps = deeper circuit
qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=2
)

# Wrap QAOA so it can solve the QuadraticProgram
meo = MinimumEigenOptimizer(qaoa)


# -----------------------------
# 3. Solve
# -----------------------------
result = meo.solve(qp)

print("\nBest binary solution found:")
for i, bit in enumerate(result.x):
    print(f"x{i} = {int(bit)}")

print("\nObjective value:", result.fval)
print("Selected coverages:", [i for i, bit in enumerate(result.x) if round(bit) == 1])

Quadratic Program:
Problem name: toy_insurance_qaoa_10

Maximize
  8*x0 + 6*x1 + 5*x2 + 9*x3 + 7*x4 + 4*x5 + 10*x6 + 3*x7 + 6*x8 + 8*x9

Subject to
  Linear constraints (1)
    x0 + x1 + x2 + x3 + x4 + x5 + x6 + x7 + x8 + x9 == 3  'pick_3'

  Binary variables (10)
    x0 x1 x2 x3 x4 x5 x6 x7 x8 x9



C:\Users\cayma\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
C:\Users\cayma\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\linalg\_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
C:\Users\cayma\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



Best binary solution found:
x0 = 1
x1 = 0
x2 = 0
x3 = 0
x4 = 1
x5 = 0
x6 = 1
x7 = 0
x8 = 0
x9 = 0

Objective value: 25.0
Selected coverages: [0, 4, 6]
